# Импорты

## Параметры синхронизации

In [ ]:
!pip install -q papermill

import papermill as pm
import pandas as pd
import gdown
import os

In [ ]:
if 'input_filename' not in locals():
    input_filename = '/content/data_1_2_well_cleanedv2.xlsx' # имя файла по умолчанию для ручного запуска
    print(f"Запуск вручную. Используем файл: {input_filename}")
else:
    print(f"Пайплайн запущен! Обрабатываем файл: {input_filename}")

if os.path.exists(input_filename):
    df = pd.read_excel(input_filename)
    print("Файл успешно загружен в DataFrame")
else:
    print(f"Ошибка! Файла {input_filename} нет в папке /content/")

In [ ]:
if 'input_filename_dates' not in locals():
    input_filename_dates = '/content/Дата_заполнения_2.xlsx' # имя файла по умолчанию для ручного запуска
    print(f"Запуск вручную. Используем файл: {input_filename_dates}")
else:
    print(f"Пайплайн запущен! Обрабатываем файл: {input_filename_dates}")

if os.path.exists(input_filename_dates):
    df_dates = pd.read_excel(input_filename_dates)
    print("Файл успешно загружен в DataFrame")
else:
    print(f"Ошибка! Файла {input_filename_dates} нет в папке /content/")

## Другие импорты

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
from datetime import datetime

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)
np.set_printoptions(suppress=True, precision=4)

In [ ]:
#df_dates = pd.read_excel("/content/Дата_заполнения_2.xlsx")
df_dates.head(3)

In [ ]:
#df = pd.read_excel("/content/data_1_2_well_cleanedv2.xlsx")
df.head(3)

In [ ]:
df = pd.merge(df, df_dates, on='id', how='left')
df.head()

# Обработка даты

In [ ]:
import pandas as pd

df['timestamp'] = pd.to_datetime(df['дата время ответа'])

df['date'] = df['timestamp'].dt.date

df['time'] = df['timestamp'].dt.time

df['month_period'] = df['timestamp'].dt.to_period('M')

In [ ]:
df.head(1)

### Добавление новых переменных

In [ ]:
max_hours = df['avr_work_hours'].max()
bins = list(range(0, int(max_hours) + 10, 10))

labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)]

df['avr_work_hours_grouped'] = pd.cut(
    df['avr_work_hours'],
    bins=bins,
    labels=labels,
    right=False
)

print(df[['avr_work_hours', 'avr_work_hours_grouped']].head())

   avr_work_hours avr_work_hours_grouped
0           44.00                  40-50
1           22.50                  20-30
2           22.50                  20-30
3           22.50                  20-30
4           22.50                  20-30


In [ ]:

burnout_mapping = {
    'Нет риска': 0,
    'Риск выгорания': 1,
    'Очень высокий риск выгорания': 2
}

df['burnout1_desc'] = df['burnout1_desc'].map(burnout_mapping)

print(df['burnout1_desc'].value_counts())

burnout1_desc
0    8337
1    1390
2    1286
Name: count, dtype: int64


### Графики

In [ ]:
def plot_burnout_dynamics(df, group_col='month_period', value_col='burnout1_desc', title='Динамика уровня выгорания (по месяцам)'):
    """
    Визуализирует динамику парамтра в виде накопленной столбчатой диаграммы.
    """
    counts = df.groupby([group_col, value_col]).size().unstack(fill_value=0)
    totals = counts.sum(axis=1)
    relative_counts = counts.divide(totals, axis=0) * 100


    fig, ax = plt.subplots(figsize=(12, 7))
    relative_counts.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        colormap='RdYlGn_r',
        width=0.7,
        edgecolor='white',
        linewidth=0.5
    )


    for i, total in enumerate(totals):
        ax.text(i, 103, f'$n={int(total)}$', ha='center', fontsize=10, fontweight='bold')


    for p in ax.patches:
        h = p.get_height()
        if h > 7:
            ax.text(
                p.get_x() + p.get_width()/2,
                p.get_y() + h/2,
                f'{int(h)}%',
                ha='center', va='center', color='black', fontsize=9
            )

    ax.set_title(title, fontsize=15, pad=20)
    ax.set_ylabel('Доля сотрудников', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylim(0, 110)
    plt.xticks(rotation=0)

    ax.legend(
        title='Риск выгорания',
        labels=['Низкий (0)', 'Средний (1)', 'Высокий (2)'],
        bbox_to_anchor=(1, 1),
        loc='upper left',
        frameon=False
    )

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)

    plt.tight_layout()
    return fig, ax

fig, ax = plot_burnout_dynamics(df)
plt.show()

In [ ]:
def check_monthly_degradation(df, target_col, date_col='date'):
    periods = sorted(df['month_period'].unique())
    report_data = []

    for i in range(1, len(periods)):
        prev_m = periods[i-1]
        curr_m = periods[i]

        s_prev = df[df['month_period'] == prev_m][target_col].dropna()
        s_curr = df[df['month_period'] == curr_m][target_col].dropna()

        # Пропускаем, если данных слишком мало для теста
        if len(s_curr) < 5 or len(s_prev) < 5:
            continue

        # Расчет средних
        m_prev = s_prev.mean()
        m_curr = s_curr.mean()
        delta = m_curr - m_prev

        # гипотеза, что в текущем месяце значения больше или равны предыдущим
        stat, p_val = stats.mannwhitneyu(s_curr, s_prev, alternative='less')

        report_data.append({
            "Текущий месяц": str(curr_m),
            "Предыдущий месяц": str(prev_m),
            "Среднее (Предыдущий месяц)": round(m_prev, 3),
            "Среднее (Текущий месяц)": round(m_curr, 3),
            "Абс. изменение": round(delta, 2),
            "% изменение": f"{(delta / m_prev * 100):+.1f}%" if m_prev != 0 else "0%",
            "p-value": round(p_val, 3),
            "Сигнал": "⚠️ Изменение" if p_val <= 0.05 else "Стабильно"
        })

    return pd.DataFrame(report_data)



In [ ]:
check_monthly_degradation(df, 'state_at_the_moment')

In [ ]:

degradation_report = check_monthly_degradation(df, 'burnout1_desc')

In [ ]:
degradation_report

# Печать отчета

In [ ]:

def add_table_page(pdf, df, title="Таблица", highlight_p_val=False):
    """Вспомогательная функция для добавления страницы с таблицей."""
    fig, ax = plt.subplots(figsize=(20, 8))
    ax.axis('off')
    ax.set_title(title, fontsize=14, pad=20)

    the_table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        loc='center',
        cellLoc='center'
    )
    the_table.auto_set_font_size(False)
    the_table.set_fontsize(9)
    the_table.scale(1.1, 2.0)

    if highlight_p_val and 'p-value' in df.columns:
        p_idx = list(df.columns).index('p-value')
        for i in range(len(df)):
            try:
                val = float(df.iloc[i, p_idx])
                if val < 0.05:
                    the_table[(i + 1, p_idx)].set_facecolor("#ffcccc")
            except (ValueError, TypeError):
                continue

    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

def save_report_to_pdf(filename, contents):
    """
    Универсальная функция для создания PDF.
    """
    with PdfPages(filename) as pdf:
        for item in contents:
            item_type = item[0]

            if item_type == 'plot':
                fig = item[1]
                pdf.savefig(fig)
                plt.close(fig)

            elif item_type == 'table':
                df_to_plot = item[1]
                title = item[2] if len(item) > 2 else "Таблица"
                highlight = item[3] if len(item) > 3 else False
                add_table_page(pdf, df_to_plot, title, highlight)

    print(f"Репорт успешно сохранен в файл: {filename}")

In [ ]:

def plot_burnout_dynamics(df, group_col='month_period', value_col='burnout1_desc',
                          title='Динамика уровня выгорания', reverse_cmap=False, custom_labels=None):

    counts = df.groupby([group_col, value_col]).size().unstack(fill_value=0)
    counts = counts.reindex(columns=sorted(counts.columns))
    totals = counts.sum(axis=1)
    relative_counts = counts.divide(totals, axis=0) * 100


    cmap_name = 'RdYlGn' if reverse_cmap else 'RdYlGn_r'
    cmap = plt.get_cmap(cmap_name)
    colors = [cmap(i) for i in np.linspace(0, 1, len(counts.columns))]

    fig, ax = plt.subplots(figsize=(12, 7))
    relative_counts.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors,
        width=0.7,
        edgecolor='white',
        linewidth=0.5
    )

    # Аннотации
    for i, total in enumerate(totals):
        ax.text(i, 103, f'$n={int(total)}$', ha='center', fontsize=10, fontweight='bold')

    for p in ax.patches:
        h = p.get_height()
        if h > 7:
            ax.text(
                p.get_x() + p.get_width()/2,
                p.get_y() + h/2,
                f'{int(h)}%',
                ha='center', va='center', color='black', fontsize=9
            )

    #ЛЕГЕНДЫ
    unique_vals = list(counts.columns)

    fig.subplots_adjust(bottom=0.25, top=0.9, left=0.1, right=0.95)

    if custom_labels and len(custom_labels) >= 2:
        label_min, label_max = custom_labels[0], custom_labels[-1]
    else:
        label_min, label_max = str(unique_vals[0]), str(unique_vals[-1])

    # ЛЕГЕНДА Границы
    elements_boundaries = [
        Line2D([0], [0], marker='s', color='w', label=label_min,
               markerfacecolor=colors[0], markersize=10),
        Line2D([0], [0], marker='s', color='w', label=label_max,
               markerfacecolor=colors[-1], markersize=10)
    ]


    leg1 = ax.legend(
        handles=elements_boundaries,
        title='',
        loc='upper center',
        bbox_to_anchor=(0.5, -0.15),
        ncol=2,
        frameon=False,
        title_fontsize=10,
        columnspacing=2
    )
    ax.add_artist(leg1)

    # ЛЕГЕНДА Реальные значения
    elements_all = [
        Line2D([0], [0], marker='s', color='w', label=str(val),
               markerfacecolor=colors[i], markersize=8)
        for i, val in enumerate(unique_vals)
    ]

    ax.legend(
        handles=elements_all,
        title='Значения:',
        loc='upper center',
        bbox_to_anchor=(0.5, -0.25),
        ncol=min(len(unique_vals), 6),
        frameon=False,
        fontsize=9,
        title_fontsize=9,
        columnspacing=1.5
    )


    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.xticks(rotation=0)

    ax.set_title(title, fontsize=15, pad=20)
    ax.set_ylabel('Доля сотрудников', fontsize=11)
    ax.set_xlabel(' ')
    ax.set_ylim(0, 110)
    plt.xticks(rotation=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)

    plt.tight_layout()
    return fig, ax

In [ ]:
df['avr_work_hours_cat'] = pd.cut(
    df['avr_work_hours'],
    bins=[0, 20, 40, 60, 80],
    labels=['0-20', '20-40', '40-60', '60+']
)

df['wage_rate_temp'] = np.where(df['wage_rate'] <= 1, 1, df['wage_rate'])

Наполнение отчета

In [ ]:

table_dynamics_burnout1_desc, _ = plot_burnout_dynamics(df, group_col='month_period', value_col='burnout1_desc', title = 'Динамика уровня выгорания по месяцам',
    reverse_cmap=False, custom_labels=['Низкий риск выгорания', 'Риск выгорания', 'Высокий риск выгорания']
)

table_dynamics_satisfaction, _ = plot_burnout_dynamics(df, group_col='month_period', value_col='satisfaction', title = 'Динамика удовлетворенности работой по месяцам',
    reverse_cmap=True, custom_labels=['Совершенно не удовлетворен работой', 'Полностью удовлетворен работой']
)

table_dynamics_state_at_the_moment, _ = plot_burnout_dynamics(df, group_col='month_period', value_col='state_at_the_moment', title = 'Динамика эмоционального состояния по месяцам',
    reverse_cmap=True, custom_labels=['Чувствую себя ощутимо хуже', 'Чувствую себя Ощутимо лучше']
)

table_dynamics_avr_work_hours_cat, _ = plot_burnout_dynamics(df, group_col='month_period', value_col='avr_work_hours_cat', title = 'Динамика средней нагрузки в часах в неделю',
    reverse_cmap=False, custom_labels=['Низкая нагрузка до 20 часов', 'Очень высокая нагрузка 60+ часов']
)

table_dynamics_wage_rate, _ = plot_burnout_dynamics(df, group_col='month_period', value_col='wage_rate_temp', title = 'Динамика средней нагрузки в совмещаемых ставках',
    reverse_cmap=False, custom_labels=['1 ставка или меньше', '4 ставки']
)


# структура отчета
report_structure = [
    ('table', check_monthly_degradation(df, 'burnout1_desc'), 'Динамика уровня выгорания по месяцам (month to month)', True),
    ('plot', table_dynamics_burnout1_desc),
    ('table', check_monthly_degradation(df, 'satisfaction'), 'Динамика удовлетворенности работой по месяцам (month to month). Вопрос - Насколько Вы удовлетворены своей работой?', True),
    ('plot', table_dynamics_satisfaction),
    ('table', check_monthly_degradation(df, 'state_at_the_moment'), 'Динамика эмоционального состояния по месяцам (month to month). Вопрос - Как Вы ощущаете себя прямо сейчас по сравнению с обычным эмоциональным состоянием?', True),
    ('plot', table_dynamics_state_at_the_moment),
    ('table', check_monthly_degradation(df, 'avr_work_hours'), 'Динамика нагрузки в часах (month to month).', True),
    ('plot', table_dynamics_avr_work_hours_cat),
    ('table', check_monthly_degradation(df, 'wage_rate'), 'Динамика нагрузки в совмещаемых ставках (month to month).', True),
    ('plot', table_dynamics_wage_rate),
]

/tmp/ipykernel_2640/2118701015.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = df.groupby([group_col, value_col]).size().unstack(fill_value=0)


Репорт успешно сохранен в файл: report_sample_final17.pdf


In [ ]:
today = str(datetime.now().strftime('%d_%m_%Y'))

output_filename = 'analytical_report_' + today + '.pdf'
print(output_filename)

preprocessing_result_11_05_2026.xlsx


In [ ]:
save_report_to_pdf(output_filename, report_structure)

print(f"Файл успешно сохранен как: {output_filename}")